# Puntos de Calor (Incendios) – IDEAM
**Fuente:** IDEAM – Sistema de Monitoreo de Puntos de Calor  
**URL:** https://puntosdecalor.ideam.gov.co/  
**Destino:** `incendios diarios.xlsx`

In [32]:
import requests
import pandas as pd
import io
from datetime import date, timedelta
from pathlib import Path

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


## 1. Configuración

In [33]:
# --- Configuración ---
TARGET_DATE = date.today() - timedelta(days=1)   # ayer (último día disponible)
DATE_STR    = TARGET_DATE.strftime("%Y-%m-%d")

REGION   = "colombia"
EXTENT   = "11.781325296112277_-86.94580078125_-1.8234225930141486_-65.43457031250001"

BASE_URL     = "https://puntosdecalor.ideam.gov.co/"
DOWNLOAD_URL = f"{BASE_URL}download-result/"

OUTPUT_DIR  = Path(r"C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output4\Indicadores\Incendios")
OUTPUT_FILE = OUTPUT_DIR / "incendios diarios.xlsx"

print(f"Fecha objetivo : {DATE_STR}")
print(f"Archivo destino: {OUTPUT_FILE}")

Fecha objetivo : 2026-04-05
Archivo destino: C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output4\Indicadores\Incendios\incendios diarios.xlsx


## 2. Descarga del CSV desde IDEAM

El endpoint `/download-result/` del servidor Django de IDEAM lee los parámetros de filtro desde el encabezado `Referer`. Se construye la URL de referencia con `from_date`, `to_date`, `region` y `extent`, y se envía como cabecera HTTP.

In [34]:
# El endpoint lee los parámetros de filtro del encabezado Referer
referer = (
    f"{BASE_URL}?from_date={DATE_STR}&to_date={DATE_STR}"
    f"&region={REGION}&extent=({EXTENT})"
)

headers = {
    "Referer": referer,
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Accept": "text/csv,application/octet-stream,*/*",
    "Accept-Language": "es-CO,es;q=0.9,en;q=0.8",
}

print(f"Descargando puntos de calor para: {DATE_STR}")
print(f"Endpoint: {DOWNLOAD_URL}")
print(f"Referer : {referer}\n")

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

resp = requests.get(DOWNLOAD_URL, headers=headers, timeout=60, verify=False)
resp.raise_for_status()

print(f"Status       : {resp.status_code}")
print(f"Content-Type : {resp.headers.get('Content-Type', 'N/A')}")
print(f"Bytes recibidos: {len(resp.content):,}")
print("\nPrimeros 400 caracteres de la respuesta:")
print(resp.text[:400])

Descargando puntos de calor para: 2026-04-05
Endpoint: https://puntosdecalor.ideam.gov.co/download-result/
Referer : https://puntosdecalor.ideam.gov.co/?from_date=2026-04-05&to_date=2026-04-05&region=colombia&extent=(11.781325296112277_-86.94580078125_-1.8234225930141486_-65.43457031250001)

Status       : 200
Content-Type : text/csv
Bytes recibidos: 24,390

Primeros 400 caracteres de la respuesta:
Fecha (UTC-5);Lat;Lon;Fuente;Temperatura (C);Temperatura Alt* (C);RadiaciÃ³n tÃ©rmica (MW);Confianza;Captura (Dia-Noche);Scan - real pixel size (km);Track - real pixel size (km)
2026-04-05 00:23;5,93055;-68,479;VIIRS-NOAA-20;35,7;11,0;2,1;Nominal;N;0,47;0,64
2026-04-05 00:23;6,01334;-69,37994;VIIRS-NOAA-20;29,0;12,3;1,1;Nominal;N;0,54;0,68
2026-04-05 00:23;6,00709;-69,38087;VIIRS-NOAA-20;45,5;1


## 3. Parseo del CSV

In [35]:
# Auto-detectar separador y mostrar todas las columnas reales
import csv

sample = resp.text[:2000]
dialect = csv.Sniffer().sniff(sample, delimiters=",;\t|")
sep_detectado = dialect.delimiter
print(f"Separador detectado: {repr(sep_detectado)}")
print(f"Primeras 2 líneas del CSV:")
for line in resp.text.splitlines()[:2]:
    print(line[:200])
print()

df_new = pd.read_csv(
    io.StringIO(resp.text),
    sep=sep_detectado,
    decimal=",",
    encoding="utf-8",
)

# Eliminar columnas Unnamed completamente vacías
df_new = df_new.loc[:, ~df_new.columns.str.startswith("Unnamed")]

# Insertar columna de fecha de descarga al inicio
df_new.insert(0, "fecha_descarga", pd.to_datetime(DATE_STR))

print(f"Filas descargadas : {len(df_new):,}")
print(f"\nColumnas ({len(df_new.columns)}):")
for c in df_new.columns:
    print(f"  {c}")
df_new.head()

Separador detectado: ';'
Primeras 2 líneas del CSV:
Fecha (UTC-5);Lat;Lon;Fuente;Temperatura (C);Temperatura Alt* (C);RadiaciÃ³n tÃ©rmica (MW);Confianza;Captura (Dia-Noche);Scan - real pixel size (km);Track - real pixel size (km)
2026-04-05 00:23;5,93055;-68,479;VIIRS-NOAA-20;35,7;11,0;2,1;Nominal;N;0,47;0,64

Filas descargadas : 291

Columnas (12):
  fecha_descarga
  Fecha (UTC-5)
  Lat
  Lon
  Fuente
  Temperatura (C)
  Temperatura Alt* (C)
  RadiaciÃ³n tÃ©rmica (MW)
  Confianza
  Captura (Dia-Noche)
  Scan - real pixel size (km)
  Track - real pixel size (km)


,fecha_descarga,Fecha (UTC-5),Lat,Lon,Fuente,Temperatura (C),Temperatura Alt* (C),RadiaciÃ³n tÃ©rmica (MW),Confianza,Captura (Dia-Noche),Scan - real pixel size (km),Track - real pixel size (km)
0,2026-04-05,2026-04-05 00:23,5.93055,-68.47900,VIIRS-NOAA-20,35.7,11.0,2.1,Nominal,N,0.47,0.64
1,2026-04-05,2026-04-05 00:23,6.01334,-69.37994,VIIRS-NOAA-20,29.0,12.3,1.1,Nominal,N,0.54,0.68
2,2026-04-05,2026-04-05 00:23,6.00709,-69.38087,VIIRS-NOAA-20,45.5,12.4,2.9,Nominal,N,0.54,0.68
3,2026-04-05,2026-04-05 00:23,5.33897,-71.84872,VIIRS-NOAA-20,42.2,7.7,3.3,Nominal,N,0.78,0.78
4,2026-04-05,2026-04-05 00:23,4.47373,-70.70148,VIIRS-NOAA-20,32.0,-1.0,1.4,Nominal,N,0.64,0.72


## 4. Carga del Excel existente y verificación de duplicados

In [36]:
if OUTPUT_FILE.exists():
    df_existing = pd.read_excel(OUTPUT_FILE, engine="openpyxl")
    print(f"Filas existentes en el archivo: {len(df_existing):,}")

    # Si el archivo no tiene columna fecha_descarga, agregarla con NaT
    if "fecha_descarga" not in df_existing.columns:
        print("Columna 'fecha_descarga' no encontrada → se agrega con valor NaT.")
        df_existing.insert(0, "fecha_descarga", pd.NaT)

    df_existing["fecha_descarga"] = pd.to_datetime(df_existing["fecha_descarga"], errors="coerce")

    # Verificar si esta fecha ya fue descargada
    fechas_existentes = df_existing["fecha_descarga"].dropna().dt.date.unique()
    if TARGET_DATE in fechas_existentes:
        print(f"AVISO: La fecha {DATE_STR} ya existe. No se agregarán duplicados.")
        df_new_clean = pd.DataFrame(columns=df_existing.columns)  # vacío
    else:
        df_new_clean = df_new.copy()
        print(f"Fecha {DATE_STR} nueva → se agregarán {len(df_new_clean):,} filas.")
else:
    print("Archivo no encontrado → se creará uno nuevo.")
    df_existing  = pd.DataFrame()
    df_new_clean = df_new.copy()

Filas existentes en el archivo: 291
AVISO: La fecha 2026-04-05 ya existe. No se agregarán duplicados.


## 5. Concatenar y guardar en Excel

In [37]:
if len(df_new_clean) > 0:
    df_combined = pd.concat([df_existing, df_new_clean], ignore_index=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    df_combined.to_excel(OUTPUT_FILE, index=False, engine="openpyxl")
    print(f"Guardado: {OUTPUT_FILE}")
    print(f"Total filas en archivo: {len(df_combined):,}")
else:
    print("No hay datos nuevos para guardar.")

No hay datos nuevos para guardar.


## 6. Estadísticas de resumen

In [38]:
if len(df_new) > 0:
    print("=" * 55)
    print(f"RESUMEN – PUNTOS DE CALOR {DATE_STR}")
    print("=" * 55)
    print(f"  Total puntos : {len(df_new):,}")

    # Distribución por fuente
    fuente_col = next((c for c in df_new.columns if "fuente" in c.lower()), None)
    if fuente_col:
        print(f"\nDistribución por fuente ({fuente_col}):")
        print(df_new[fuente_col].value_counts().to_string())

    # Distribución por confianza
    conf_col = next((c for c in df_new.columns if "confianza" in c.lower()), None)
    if conf_col:
        print(f"\nDistribución por confianza ({conf_col}):")
        print(df_new[conf_col].value_counts().to_string())

    # Distribución día/noche
    dn_col = next((c for c in df_new.columns
                   if any(k in c.lower() for k in ["dia", "noche", "captura"])), None)
    if dn_col:
        print(f"\nCaptura día/noche ({dn_col}):")
        print(df_new[dn_col].value_counts().to_string())

    # Temperatura y radiación promedio
    temp_col = next((c for c in df_new.columns if "temp" in c.lower()), None)
    rad_col  = next((c for c in df_new.columns
                     if any(k in c.lower() for k in ["rad", "frp", "potencia"])), None)
    if temp_col:
        try:
            print(f"\nTemperatura ({temp_col}): promedio = {pd.to_numeric(df_new[temp_col], errors='coerce').mean():.1f} K")
        except Exception:
            pass
    if rad_col:
        try:
            print(f"Radiación/FRP ({rad_col}): promedio = {pd.to_numeric(df_new[rad_col], errors='coerce').mean():.2f} MW")
        except Exception:
            pass
else:
    print(f"No se encontraron puntos de calor para {DATE_STR}.")

RESUMEN – PUNTOS DE CALOR 2026-04-05
  Total puntos : 291

Distribución por fuente (Fuente):
Fuente
VIIRS-NOAA-21      103
VIIRS-Suomi-NPP     78
VIIRS-NOAA-20       69
MODIS-Aqua          37
MODIS-Terra          4

Distribución por confianza (Confianza):
Confianza
Nominal    218
Alta        17
Baja        15
77 %         2
59 %         2
61 %         2
78 %         2
69 %         2
48 %         2
68 %         2
55 %         2
0 %          2
73 %         2
60 %         2
57 %         2
72 %         1
66 %         1
82 %         1
47 %         1
23 %         1
18 %         1
45 %         1
34 %         1
89 %         1
85 %         1
58 %         1
67 %         1
62 %         1
70 %         1
83 %         1
51 %         1
38 %         1

Captura día/noche (RadiaciÃ³n tÃ©rmica (MW)):
RadiaciÃ³n tÃ©rmica (MW)
6.0     8
6.4     8
1.1     6
1.5     6
5.2     6
5.4     6
6.1     5
4.3     5
4.1     5
2.9     5
4.7     5
5.0     5
5.9     5
3.1     4
1.2     4
3.7     4
4.0     4
3.5     4
2.